# 🎬 AutoVideo - Kaggle Notebook Setup (Fixed Cloudflare DNS)

Auto Recap & Video Translation API on Kaggle GPU Instance with Public Web Tunnel.

### ⚠️ Required Settings:
1. **Accelerator**: GPU T4 or P100
2. **Internet**: ON

In [ ]:
# 1. Git Clone Project from GitHub & Install Dependencies
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg

import os
REPO_URL = "https://github.com/surviveman78-commits/autovideo.git"

if not os.path.exists('/kaggle/working/autovideo'):
    !git clone {REPO_URL} /kaggle/working/autovideo

%cd /kaggle/working/autovideo
!pip install -q -r requirements.txt

In [ ]:
# 2. Setup VoxCPM Repository for GPU Voice Clone
import os
if not os.path.exists('VoxCPM'):
    !git clone https://github.com/OpenBMB/VoxCPM.git

print("Current Directory:", os.getcwd())
print("Files:", os.listdir('.'))

In [ ]:
# 3. Set API Keys (Optional - Can also be set directly in Web UI)
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"
print("API Keys set in environment!")

In [ ]:
# 4. Launch FastAPI Web App & Cloudflare Tunnel with DNS Wait
import os
import re
import time
import subprocess
import sys

if not os.path.exists("./cloudflared"):
    print("📥 Downloading Cloudflare Tunnel Binary...")
    subprocess.run(["curl", "-L", "--output", "cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

print("🚀 Starting AutoVideo FastAPI Server...")
with open("uvicorn.log", "w") as uvicorn_log:
    server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=uvicorn_log,
        stderr=uvicorn_log
    )
time.sleep(4)

print("🌐 Opening Public Web Tunnel...")
with open("cloudflared.log", "w") as cf_log:
    tunnel_process = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=cf_log,
        stderr=cf_log
    )

print("⏳ Establishing tunnel & registering DNS (waiting 6s)...")
time.sleep(6)

public_url = None
if os.path.exists("cloudflared.log"):
    with open("cloudflared.log", "r") as f:
        content = f.read()
        urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if urls:
            public_url = urls[-1]

if public_url:
    print("\n" + "="*70)
    print("🎉 AUTOVIDEO WEB UI IS LIVE AT:")
    print(f"👉 {public_url}")
    print("="*70)
    print("⚠️ NOTE: If you see 'DNS_PROBE_FINISHED_NXDOMAIN', wait 10-15s and refresh!")
    print("="*70 + "\n")
else:
    print("⚠️ Check cloudflared.log below:")
    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            print(f.read())

In [ ]:
# 5. Check Rendered Videos & Download
import os
from IPython.display import FileLink, display

recap_dir = "downloads/recap"
if os.path.exists(recap_dir):
    for f in os.listdir(recap_dir):
        file_path = os.path.join(recap_dir, f)
        print(f"- {f} ({os.path.getsize(file_path)/(1024*1024):.2f} MB)")
        display(FileLink(file_path))
else:
    print("No rendered videos found yet.")